# AgentCore Gateway — Streamable HTTP

## 개요

MCP 응답 스트리밍을 사용하면 AgentCore Gateway가 도구 실행 중 클라이언트에 실시간 Server-Sent Events(SSE)를 전달할 수 있습니다. 전체 도구 호출이 완료될 때까지 기다린 후 응답을 반환하는 대신, Gateway는 이벤트가 발생하는 즉시 스트리밍합니다. 여기에는 [진행 상황 알림](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-mcp-progress.html), [로그 메시지](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-mcp-logging.html), [정보 요청](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-mcp-elicitation.html), [샘플링](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-mcp-sampling.html) 요청이 포함됩니다.

![스트리밍 다이어그램](./images/streaming.png)


응답 스트리밍을 활성화하려면 AgentCore Gateway를 생성하거나 업데이트할 때 `protocolConfiguration.mcp` 필드의 `streamingConfiguration.enableResponseStreaming`을 `true`로 설정합니다.

```bash


{
  "protocolConfiguration": {
    "mcp": {
      "streamingConfiguration": {
        "enableResponseStreaming": true
      }
    }
  }
}
```

참고: 응답 스트리밍을 활성화하면 응답 인터셉터의 입력 계약이 변경됩니다. 응답 인터셉터를 사용하는 경우 스트리밍 응답과 호환되는지 인터셉터 로직을 검토하세요. 자세한 내용은 스트리밍이 활성화된 경우의 [응답 인터셉터](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-interceptors-types.html#gateway-interceptors-types-streaming)를 참조하세요.

## 응답 스트리밍 작동 방식

응답 스트리밍이 활성화된 상태에서 클라이언트가 Accept: text/event-stream 요청을 보내면 Gateway는 단일 JSON 응답 대신 SSE 스트림을 반환합니다. 이벤트는 MCP server target에서 수신되는 즉시 전달됩니다.

클라이언트가 Accept: text/event-stream을 보내지 않으면 Gateway는 응답을 버퍼링하고 도구 호출이 완료된 후 단일 JSON 응답을 반환합니다. 이 경우 중간 이벤트(진행 상황, 로깅)는 전달되지 않습니다.


## 워크숍 로드맵

| 단계 | 수행 내용 |
|---|---|
| **1** | Notebook을 설정합니다(환경 변수, 유틸리티, 로깅). |
| **2** | Gateway를 생성합니다(Cognito 인바운드 인증, IAM 역할, 스트리밍이 활성화된 Gateway). |
| **3** | `labstream` FastMCP server를 AgentCore Runtime에 배포합니다. |
| **4** | Gateway target으로 연결합니다(아웃바운드 OAuth, target 생성, 인바운드 토큰). |
| **5** | 하위 호환성을 확인합니다. `Accept: application/json`은 단일 JSON 응답만 반환합니다. |
| **6** | 서버에서 전송하는 진행 상황 알림을 확인합니다(`streaming_demo`). |
| **7** | 스트림 중간에 발생하는 도구 예외를 확인합니다(`failing_demo`). |
| **8** | 서버에서 전송하는 로그 이벤트를 확인합니다(`logging_demo`). |
| **9** | 30초 간격의 진행 상황 알림을 이용한 장기 실행 keep-alive를 확인합니다(`keepalive_demo`). |
| **10** | 리소스를 정리합니다. |

## 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:---|:---|
| 튜토리얼 유형 | 대화형 |
| AgentCore 구성 요소 | AgentCore Gateway, AgentCore Identity, AgentCore Runtime |
| Gateway target 유형 | MCP server |
| Gateway 기능 | 스트리밍 켜짐, 세션 꺼짐, 인터셉터 없음 |
| MCP 전송 방식 | Streamable HTTP, SSE |
| 인바운드 인증 | Cognito (M2M) |
| 아웃바운드 인증 | OAuth2 credential provider를 통한 Cognito (M2M) |
| 사용 SDK | boto3 + SSE용 원시 httpx |


### 1단계: 설정 및 사전 요구 사항

Jupyter(Python 3.10 이상 커널), Node.js + npm(AgentCore CLI용), 그리고 CloudFormation, Cognito IDP, IAM, Bedrock AgentCore(제어 + 런타임)에 대한 IAM 권한이 필요합니다.

In [ ]:
# 현재 디렉터리의 requirements.txt 또는 pyproject.toml에서 설치
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
!npm install -g @aws/agentcore

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 이 Notebook에서 사용할 import와 상수
import utils
import logging
import boto3
import json

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

REGION = boto3.Session().region_name
COGNITO_STACK_NAME = "agentcore-gateway-lab"
TEMPLATE_PATH = "cloudformation/cognito-signup-stack.yaml"
MCP_SERVER_NAME = "lab4stream"
GATEWAY_NAME = "ac-gateway-streaming-only"

cfn = boto3.client("cloudformation", region_name=REGION)
cognito = boto3.client("cognito-idp", region_name=REGION)
print("REGION:", REGION)

### 2단계: Gateway 생성

### 2.1단계: CloudFormation으로 Cognito 배포

멱등성을 보장합니다. 기존 `agentcore-gateway-lab` 스택이 이미 배포되어 있으면(예: `01-mcp-server-target.ipynb`에서 배포) 해당 스택을 재사용합니다.

In [ ]:
outputs = utils.deploy_cognito_stack(cfn, COGNITO_STACK_NAME, TEMPLATE_PATH)

# Gateway 인바운드
gw_user_pool_id = outputs["UserPoolId"]
gw_client_id = outputs["GatewayClientId"]
gw_cognito_discovery_url = outputs["DiscoveryUrl"]
scopeString = outputs["GatewayScope"]
token_endpoint = outputs["TokenEndpoint"]
gw_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=gw_client_id
)["UserPoolClient"]["ClientSecret"]

# MCP server로의 아웃바운드(동일한 풀)
runtime_client_id = outputs["MCPClientId"]
runtime_cognito_discovery_url = gw_cognito_discovery_url
runtimeScopeString = outputs["MCPScope"]
runtime_client_secret = cognito.describe_user_pool_client(
    UserPoolId=gw_user_pool_id, ClientId=runtime_client_id
)["UserPoolClient"]["ClientSecret"]

print(f"User Pool ID:       {gw_user_pool_id}")
print(f"Discovery URL:      {gw_cognito_discovery_url}")
print(f"Token endpoint:     {token_endpoint}")
print(f"Gateway client ID:  {gw_client_id}")
print(f"MCP client ID:      {runtime_client_id}")
print(f"Gateway scope:      {scopeString}")
print(f"MCP scope:          {runtimeScopeString}")

### 2.2단계: Gateway IAM 역할 생성

In [ ]:
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role_with_region(
    GATEWAY_NAME, REGION
)
print("AgentCore Gateway role ARN:", agentcore_gateway_iam_role["Role"]["Arn"])

### 2.3단계: 스트리밍 전용 구성으로 Gateway 생성

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [gw_client_id],
        "discoveryUrl": gw_cognito_discovery_url,
    }
}

create_response = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=agentcore_gateway_iam_role["Role"]["Arn"],
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": ["2025-11-25"],
            "streamingConfiguration": {"enableResponseStreaming": True},
        }
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="Streaming-only gateway (no sessions, no interceptor)",
)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID:  {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

### 3단계: AgentCore Runtime에 MCP server 배포

### 3.1단계: MCP server 코드 확인

`labstream`은 스트리밍과 관련된 네 가지 도구인 `streaming_demo`, `failing_demo`, `logging_demo`, `keepalive_demo`와 하위 호환성 데모를 위한 간단한 `getOrder`를 제공합니다.

In [ ]:
from IPython.display import Code

Code("mcpservers/app/labstream/main.py", language="python")

### 3.2단계: Agent 등록

In [ ]:
!cd mcpservers && agentcore add agent \
    --name {MCP_SERVER_NAME} \
    --type byo \
    --language Python \
    --protocol MCP \
    --code-location app/labstream \
    --authorizer-type CUSTOM_JWT \
    --discovery-url {runtime_cognito_discovery_url} \
    --allowed-clients {runtime_client_id} \
    --allowed-scopes {runtimeScopeString}

### 3.3단계: AgentCore CLI로 배포

In [ ]:
!cd mcpservers && agentcore deploy

In [ ]:
agent = utils.get_agent_status(MCP_SERVER_NAME)

mcp_arn = agent["identifier"]
mcp_url = agent["invocationUrl"]
mcp_id = mcp_arn.split("/")[-1]

print(f"mcp_arn: {mcp_arn}")
print(f"mcp_id:  {mcp_id}")
print(f"mcp_url: {mcp_url}")

### 4단계: MCP server를 Gateway target으로 연결

### 4.1단계: 아웃바운드 OAuth2 credential provider

In [ ]:
identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name=f"{GATEWAY_NAME}-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": runtime_cognito_discovery_url},
            "clientId": runtime_client_id,
            "clientSecret": runtime_client_secret,
        }
    },
)
cognito_provider_arn = cognito_provider["credentialProviderArn"]
print(cognito_provider_arn)

### 4.2단계: Gateway target 생성

In [ ]:
create_gateway_target_response = gateway_client.create_gateway_target(
    name="mcp-server-target",
    gatewayIdentifier=gatewayID,
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_url}}},
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": cognito_provider_arn,
                    "scopes": [runtimeScopeString],
                }
            },
        },
    ],
    # 클라이언트가 제공한 `Mcp-Session-Id`를 양방향으로 Runtime에 전달하여
    # AgentCore Runtime이 요청을 특정 microvm에 고정할 수 있게 합니다.
    metadataConfiguration={
        "allowedRequestHeaders": ["Mcp-Session-Id"],
        "allowedResponseHeaders": ["Mcp-Session-Id"],
    },
)
gatewayTargetID = create_gateway_target_response["targetId"]
print(f"Created target: {gatewayTargetID}")

### 4.3단계: Target이 READY 상태인지 확인

In [ ]:
list_targets_response = gateway_client.list_gateway_targets(gatewayIdentifier=gatewayID)
print(json.dumps(list_targets_response, default=str, indent=2))

### 4.4단계: 인바운드 액세스 토큰 가져오기

In [ ]:
token_response = utils.get_token(
    token_endpoint, gw_client_id, gw_client_secret, scopeString
)
token = token_response["access_token"]
print("Token (truncated):", token[:60], "...")

## 5단계: 하위 호환성 — `Accept: application/json`

클라이언트가 Accept: text/event-stream을 보내지 않으면 Gateway는 응답을 버퍼링하고 도구 호출이 완료된 후 단일 JSON 응답을 반환합니다. 이 경우 중간 이벤트(진행 상황, 로깅)는 전달되지 않습니다.

In [ ]:
import uuid
from gateway_mcp_client import GatewayMCPClient


def _get_inbound_token() -> str:
    return utils.get_token(token_endpoint, gw_client_id, gw_client_secret, scopeString)[
        "access_token"
    ]


session_id = str(uuid.uuid4())

mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

In [ ]:
# 클라이언트가 제공한 `Mcp-Session-Id`입니다. Runtime은 `stateless_http=True`이므로
# 이 ID를 발급하지 않지만, target의 metadataConfiguration 허용 목록에
# Mcp-Session-Id를 추가했으므로(4.2단계) Gateway가 이 헤더를 Runtime에 전달하고
# AgentCore Runtime은 microvm 선호도에 이 헤더를 사용합니다. 호출 간에 동일한 ID를
# 재사용하면 이후 요청이 동일한 microvm 인스턴스에 계속 고정됩니다.

print(f"Client-supplied Mcp-Session-Id: {mcp.session_id}")


response = mcp.call_tool_json_only("mcp-server-target___getOrder", {}, request_id=2)
print("\n=== A) getOrder (no intermediate frames) ===")
print(f"HTTP {response['http_status']}  Content-Type: {response['content_type']}")
print(f"Body: {response['body']}")

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

buf = mcp.call_tool_json_only(
    "mcp-server-target___streaming_demo", {"steps": 5}, request_id=20
)
print("\n=== B) streaming_demo (buffered — progress events dropped) ===")
print(f"HTTP {buf['http_status']}  Content-Type: {buf['content_type']}")
print(f"Body: {buf['body']}")
print()
print(
    "Note: the body contains only the final tool result. The 5 "
    "`notifications/progress` frames the server emitted were discarded "
    "because the client did not request `text/event-stream`."
)

## 6단계: 서버에서 전송하는 진행 상황 알림

SSE 스트림 응답 예시:

```
event: message
data: {"jsonrpc":"2.0","method":"notifications/progress","params":{"progressToken":"auto-1","progress":1,"total":3,"message":"Loading data..."}}
event: message
data: {"jsonrpc":"2.0","method":"notifications/message","params":{"level":"info","logger":"analyzer","data":"Processing 10,000 records"}}
event: message
data: {"jsonrpc":"2.0","method":"notifications/progress","params":{"progressToken":"auto-1","progress":2,"total":3,"message":"Analyzing..."}}
event: message
data: {"jsonrpc":"2.0","method":"notifications/progress","params":{"progressToken":"auto-1","progress":3,"total":3,"message":"Complete"}}
event: message
data: {"jsonrpc":"2.0","id":"tool-call-1","result":{"content":[{"type":"text","text":"Analysis complete. Found 3 anomalies."}]}}```

In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print("--- streaming_demo SSE frames ---")
for msg in mcp.stream_tool_call(
    "mcp-server-target___streaming_demo",
    {"steps": 5},
    progress_token="demo-progress",
    request_id=3,
):
    print(json.dumps(msg))

## 7단계: 스트림 중간의 도구 예외

`failing_demo(steps=3)`은 진행 상황 알림 두 개를 전송한 후 `RuntimeError`를 발생시킵니다. 스트리밍 응답은 진행 상황 프레임을 전달한 다음 `result.isError=true` 콘텐츠 블록을 최종 SSE 프레임으로 전달합니다.


In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print("--- failing_demo SSE frames (3 progress, then error) ---")
for msg in mcp.stream_tool_call(
    "mcp-server-target___failing_demo",
    {"steps": 3},
    progress_token="failing-demo",
    request_id=4,
):
    print(json.dumps(msg))

## 8단계: 서버에서 전송하는 로그 이벤트

`logging_demo()`는 심각도별로 하나의 `notifications/message`를 전송합니다.


In [ ]:
mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

print("--- logging_demo SSE frames (4 log events + result) ---")
for msg in mcp.stream_tool_call(
    "mcp-server-target___logging_demo",
    {},
    request_id=5,
):
    print(json.dumps(msg))

## 9단계: 30초 간격의 진행 상황 알림을 이용한 장기 실행 keep-alive

`keepalive_demo(duration_seconds=N, interval_seconds=30, emit_progress=True)`는 `duration_seconds` 동안 대기하면서 30초마다 진행 상황 프레임 하나를 전송합니다. Gateway의 기본 요청 제한 시간인 15분을 초과하는 도구 호출에 keep-alive 패턴으로 사용합니다.

> 셀이 빠르게 완료되도록 데모는 `duration_seconds=60`으로 실행됩니다.

In [ ]:
import time

started = time.time()
n_progress = 0
saw_result = False

mcp = GatewayMCPClient(gatewayURL, _get_inbound_token, session_id=session_id)

for msg in mcp.stream_tool_call(
    "mcp-server-target___keepalive_demo",
    {"duration_seconds": 60, "interval_seconds": 30, "emit_progress": True},
    progress_token="keepalive-60s",
    request_id=6,
):
    if msg.get("method") == "notifications/progress":
        n_progress += 1
        print(f"  progress #{n_progress} at {round(time.time() - started, 1)}s")
    elif msg.get("id") == 6:
        saw_result = True
        print(f"  result at {round(time.time() - started, 1)}s: {msg.get('result')}")

print(
    f"\nelapsed={round(time.time() - started, 1)}s  "
    f"progress_count={n_progress}  result_seen={saw_result}"
)

## 10단계: 정리

아래 셀의 주석을 해제하여 이 Notebook에서 생성한 Gateway, OAuth2 credential provider, MCP server runtime, IAM 역할을 삭제합니다. Cognito CloudFormation 스택은 여러 실습에서 공유하므로 모든 실습을 마친 경우가 아니면 그대로 두세요.

In [ ]:
# utils.delete_gateway(gateway_client, gatewayID)

In [ ]:
# identity_client.delete_oauth2_credential_provider(name=f"{GATEWAY_NAME}-identity")

In [ ]:
# !cd mcpservers && agentcore remove agent --name {MCP_SERVER_NAME} -y
# !cd mcpservers && agentcore deploy -y

In [ ]:
# # ## 다른 실습에서 이 스택을 사용하지 않을 때 Cognito 스택 삭제
# print(f"Deleting stack {COGNITO_STACK_NAME}...")
# cfn.delete_stack(StackName=COGNITO_STACK_NAME)
# cfn.get_waiter("stack_delete_complete").wait(StackName=COGNITO_STACK_NAME)
# print(f"✅ Stack {COGNITO_STACK_NAME} deleted")

In [ ]:
# utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")